In [1]:
!pip install transformers==4.57.1 accelerate==1.14.0 sentencepiece==0.2.1 safetensors==0.8.0 huggingface_hub==0.36.2 langchain-huggingface==1.2.2 langchain-core==1.4.8

  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.21.0
    Uninstalling huggingface_hub-1.21.0:
      Successfully uninstalled huggingface_hub-1.21.0
  Attempting uninstall: transformers━━━━━━━━━━━━ 0/2 [huggingface_hub]
    Found existing installation: transformers 5.12.12m0/2 [huggingface_hub]
    Uninstalling transformers-5.12.1:╺━━━━━━━━━━━━━━━━━━━ 1/2 [transformers]
      Successfully uninstalled transformers-5.12.1━━━━━━━━━━━━━━━━ 1/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]


In [2]:
!pip install hf_transfer

In [3]:
!pip install -U datasets peft trl accelerate bitsandbytes "transformers==4.57.1"


In [4]:
pip install -U transformers accelerate bitsandbytes

  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.21.0-py3-none-any.whl.metadata (14 kB)
Using cached transformers-5.12.1-py3-none-any.whl (11.2 MB)
Using cached huggingface_hub-1.21.0-py3-none-any.whl (721 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers━━━━━━━━━━━━ 0/2 [huggingface-hub]
    Found existing installation: transformers 4.57.12m0/2 [huggingface-hub]
    Uninstalling transformers-4.57.1:╺━━━━━━━━━━━━━━━━━━━ 1/2 [transformers]
      Successfully uninstalled transformers-4.57.1━━━━━━━━━━━━ 1/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")

if hf_token is None:
    raise ValueError(
        "HF_TOKEN 환경변수가 없습니다. RunPod Secrets에 HF_TOKEN을 등록했는지 확인하세요."
    )

login(token=hf_token)

print("Hugging Face login completed.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login completed.


In [3]:
import json
from collections import Counter

from datasets import load_dataset


JSONL_PATH = "masking_train_500.jsonl"

LABEL_CATS = [
    "comp_name",
    "person_name",
    "address",
    "personal_info",
    "school_edu",
    "project_name",
    "jd_discrimination",
]


def _assistant_text(example):
    for message in reversed(example["messages"]):
        if message.get("role") == "assistant":
            return message.get("content", "")
    return ""


def add_strata(example):
    label = example.get("label")

    if isinstance(label, dict):
        clean = all(not label.get(category) for category in LABEL_CATS)
    else:
        try:
            obj = json.loads(_assistant_text(example))
            clean = all(not obj.get(category) for category in LABEL_CATS)
        except Exception:
            clean = False

    return {
        "strata": "clean" if clean else "sensitive"
    }


dataset = load_dataset(
    "json",
    data_files={"train": JSONL_PATH},
)["train"]

dataset = dataset.map(add_strata)
dataset = dataset.class_encode_column("strata")

names = dataset.features["strata"].names
print("total size:", len(dataset))
print("total strata:", Counter(names[i] for i in dataset["strata"]))

total size: 500
total strata: Counter({'sensitive': 400, 'clean': 100})


In [4]:
from collections import Counter
import json


OUTPUT_DIR = "./exaone-base"

LABEL_CATS = [
    "comp_name",
    "person_name",
    "address",
    "personal_info",
    "school_edu",
    "project_name",
    "jd_discrimination",
]


format_instructions = """
{
  "comp_name": [],
  "person_name": [],
  "address": [],
  "personal_info": [],
  "school_edu": [],
  "project_name": [],
  "jd_discrimination": []
}
""".strip()


STRICT_SYSTEM_PROMPT = f"""
당신은 한국 채용 데이터의 개인정보 마스킹 전문가입니다.
입력으로 들어온 자유 텍스트(자기소개서, 채용공고(JD) 본문, 회사 소개,
채용 사유, 메모 등)에서 "마스킹이 필요한 표현"을 찾아
아래 7개 카테고리로 분류하여 추출하세요.

[마스킹 카테고리]
1. comp_name        : 회사/기관/고객사/이전 근무처 등 조직 식별명
2. person_name      : 지원자 본인 및 제3자(교수·추천인·동료 등)의 실명
3. address          : 주소 및 출신·거주 지역
4. personal_info    : 고유식별정보·연락처·차별위험정보·민감정보
5. school_edu       : 학교/교육기관/주최기관명
6. project_name     : 내부·제3자 정보가 포함될 수 있는 프로젝트 실명
7. jd_discrimination: JD 내 차별 소지 조항

[출력 규칙]
- 반드시 JSON object 하나만 출력한다.
- 첫 글자는 반드시 {{ 로 시작하고 마지막 글자는 반드시 }} 로 끝난다.
- 코드블록, 코드펜스, ```json, ``` 를 절대 사용하지 않는다.
- 설명, 해설, 마크다운, 리스트 단독 출력, 번역, 영어 변환을 절대 하지 않는다.
- 입력 텍스트에 등장한 원문 표현 그대로 추출한다.
- 동일 표현은 한 번만 넣는다.
- 해당 카테고리에 없으면 빈 리스트 []를 넣는다.
- 키는 반드시 아래 JSON 스키마의 7개만 사용한다.

[출력 형식]
{format_instructions}
""".strip()


def force_system_prompt(messages, include_answer=True):
    new_messages = []

    source_messages = list(messages)
    if not include_answer and source_messages and source_messages[-1].get("role") == "assistant":
        source_messages = source_messages[:-1]

    has_system = source_messages and source_messages[0].get("role") == "system"

    if has_system:
        new_messages.append({
            "role": "system",
            "content": STRICT_SYSTEM_PROMPT,
        })
        new_messages.extend(source_messages[1:])
    else:
        new_messages.append({
            "role": "system",
            "content": STRICT_SYSTEM_PROMPT,
        })
        new_messages.extend(source_messages)

    return new_messages


# 1차 split: test 20%
split_1 = dataset.train_test_split(
    test_size=0.2,
    seed=42,
    stratify_by_column="strata",
)

train_val_dataset = split_1["train"]
test_dataset_raw = split_1["test"]

# 2차 split: 남은 80% 중 validation 25% = 전체 기준 20%
split_2 = train_val_dataset.train_test_split(
    test_size=0.25,
    seed=42,
    stratify_by_column="strata",
)

train_dataset_raw = split_2["train"]
eval_dataset_raw = split_2["test"]

names = dataset.features["strata"].names

print("train strata:", Counter(names[i] for i in train_dataset_raw["strata"]))
print("val   strata:", Counter(names[i] for i in eval_dataset_raw["strata"]))
print("test  strata:", Counter(names[i] for i in test_dataset_raw["strata"]))

remove_cols = ["strata"] + (["label"] if "label" in dataset.column_names else [])

train_dataset = train_dataset_raw.remove_columns(remove_cols)
eval_dataset = eval_dataset_raw.remove_columns(remove_cols)
test_dataset = test_dataset_raw.remove_columns(remove_cols)

print("train size:", len(train_dataset))
print("val size  :", len(eval_dataset))
print("test size :", len(test_dataset))
print("output dir:", OUTPUT_DIR)

train strata: Counter({'sensitive': 240, 'clean': 60})
val   strata: Counter({'sensitive': 80, 'clean': 20})
test  strata: Counter({'sensitive': 80, 'clean': 20})
train size: 300
val size  : 100
test size : 100
output dir: ./exaone-base


In [8]:
# ================================================================
# Base EXAONE 평가 전용 셀
# - 전제: test_dataset, STRICT_SYSTEM_PROMPT, LABEL_CATS가 이미 정의되어 있음
# - 학습 안 함
# - LoRA 안 붙임
# - base EXAONE으로 test_dataset 추론 후 metric 계산
# ================================================================

import inspect
import sys
import types
import re
import json
from typing import TypedDict

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


MODEL_NAME = "LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct"


# ---------------------------------------------------------------- compatibility patches

try:
    import transformers.modeling_rope_utils as _rope_utils

    if not hasattr(_rope_utils, "RopeParameters"):
        class RopeParameters(TypedDict, total=False):
            rope_type: str
            factor: float
            low_freq_factor: float
            high_freq_factor: float
            original_max_position_embeddings: int
            attention_factor: float
            beta_fast: float
            beta_slow: float
            short_factor: list[float]
            long_factor: list[float]

        _rope_utils.RopeParameters = RopeParameters
        print("[PATCH] Added missing transformers.modeling_rope_utils.RopeParameters shim.")
except Exception as e:
    print(f"[PATCH][WARN] RopeParameters shim skipped: {e}")


try:
    import transformers.integrations as _tf_integrations

    def _noop_kernel_patch(*args, **kwargs):
        if args and callable(args[0]) and len(args) == 1:
            return args[0]

        def _decorator(fn):
            return fn

        return _decorator

    for _name in (
        "use_kernel_forward_from_hub",
        "use_kernel_func_from_hub",
        "use_kernelized_func",
    ):
        if not hasattr(_tf_integrations, _name):
            setattr(_tf_integrations, _name, _noop_kernel_patch)
            print(f"[PATCH] Added missing transformers.integrations.{_name} shim.")
except Exception as e:
    print(f"[PATCH][WARN] integrations shim skipped: {e}")


def _patch_exaone(model):
    if hasattr(model, "_exaone_compat_patched") and model._exaone_compat_patched:
        return model

    if hasattr(model, "transformer") and hasattr(model.transformer, "wte"):
        embed = model.transformer.wte
    elif hasattr(model, "transformer") and hasattr(model.transformer, "embed_tokens"):
        embed = model.transformer.embed_tokens
    elif hasattr(model, "model") and hasattr(model.model, "embed_tokens"):
        embed = model.model.embed_tokens
    else:
        raise AttributeError(
            "embedding layer not found. Check model structure.\n" + str(model)
        )

    model.get_input_embeddings = lambda: embed
    model.set_input_embeddings = lambda v: setattr(embed, "weight", v.weight)

    import transformers.masking_utils as masking_utils

    original_create_causal_mask = masking_utils.create_causal_mask
    original_params = inspect.signature(original_create_causal_mask).parameters

    def _create_causal_mask_compat(*args, **kwargs):
        if "input_embeds" in kwargs and "input_embeds" not in original_params:
            value = kwargs.pop("input_embeds")
            if "inputs_embeds" in original_params:
                kwargs["inputs_embeds"] = value
            elif "input_tensor" in original_params:
                kwargs["input_tensor"] = value
            else:
                kwargs["input_ids"] = value
        elif "inputs_embeds" in kwargs and "inputs_embeds" not in original_params:
            value = kwargs.pop("inputs_embeds")
            if "input_embeds" in original_params:
                kwargs["input_embeds"] = value
            elif "input_tensor" in original_params:
                kwargs["input_tensor"] = value
            else:
                kwargs["input_ids"] = value

        accepts_var_kwargs = any(
            p.kind == inspect.Parameter.VAR_KEYWORD
            for p in original_params.values()
        )

        if not accepts_var_kwargs:
            kwargs = {
                key: value
                for key, value in kwargs.items()
                if key in original_params
            }

        return original_create_causal_mask(*args, **kwargs)

    masking_utils.create_causal_mask = _create_causal_mask_compat

    for module in list(sys.modules.values()):
        if module is None or not hasattr(module, "create_causal_mask"):
            continue
        if getattr(module, "create_causal_mask", None) is original_create_causal_mask:
            setattr(module, "create_causal_mask", _create_causal_mask_compat)

    backbone = getattr(model, "transformer", None) or getattr(model, "model", None)

    if backbone is not None and not hasattr(backbone, "_exaone_forward_patched"):
        original_forward = backbone.forward

        def _patched_forward(self, *args, **kwargs):
            if "input_embeds" in kwargs and "inputs_embeds" not in kwargs:
                kwargs["inputs_embeds"] = kwargs.pop("input_embeds")
            return original_forward(*args, **kwargs)

        backbone.forward = types.MethodType(_patched_forward, backbone)
        backbone._exaone_forward_patched = True

    model._exaone_compat_patched = True
    print("[PATCH] EXAONE compatibility patch applied.")
    return model


# ---------------------------------------------------------------- tokenizer + base model load

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = _patch_exaone(base_model)
base_model.eval()

if hasattr(base_model, "config"):
    base_model.config.use_cache = True

print("[LOAD] Base EXAONE loaded for evaluation only.")


# ---------------------------------------------------------------- prediction / parsing / metrics

def empty_masking_result():
    return {key: [] for key in LABEL_CATS}


def parse_prediction_json(text):
    text = text.strip()

    # 코드펜스 제거
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # 앞에 ], | 같은 이상 토큰이 붙어도 JSON object만 추출
    first = text.find("{")
    last = text.rfind("}")

    if first == -1 or last == -1 or first >= last:
        raise ValueError(f"JSON object를 찾지 못했습니다: {text}")

    text = text[first:last + 1]
    obj = json.loads(text)

    if not isinstance(obj, dict):
        return empty_masking_result()

    return {
        key: obj.get(key, []) if isinstance(obj.get(key, []), list) else []
        for key in LABEL_CATS
    }


def normalize_gold_json(text):
    obj = json.loads(text)

    return {
        key: obj.get(key, []) if isinstance(obj.get(key, []), list) else []
        for key in LABEL_CATS
    }


def item_set(values):
    return set(
        str(value).strip()
        for value in values
        if str(value).strip()
    )


def safe_div(num, den):
    return num / den if den else 0.0


def predict_base_exaone(input_json_text, model=base_model, tokenizer=tokenizer):
    messages = [
        {
            "role": "system",
            "content": STRICT_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": input_json_text,
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    device = next(model.parameters()).device

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    ).strip()


def evaluate_base_exaone(dataset_to_eval, max_samples=None, show_progress=True):
    n = len(dataset_to_eval) if max_samples is None else min(len(dataset_to_eval), max_samples)

    total_tp = 0
    total_fp = 0
    total_fn = 0

    exact_match_count = 0
    parse_fail_count = 0

    for i in range(n):
        sample = dataset_to_eval[i]

        input_text = sample["messages"][1]["content"]
        gold_text = sample["messages"][-1]["content"]

        pred_raw = predict_base_exaone(input_text)
        gold = normalize_gold_json(gold_text)

        try:
            pred = parse_prediction_json(pred_raw)
        except Exception:
            parse_fail_count += 1
            pred = empty_masking_result()

        sample_exact = True

        for key in LABEL_CATS:
            pred_items = item_set(pred.get(key, []))
            gold_items = item_set(gold.get(key, []))

            tp = len(pred_items & gold_items)
            fp = len(pred_items - gold_items)
            fn = len(gold_items - pred_items)

            total_tp += tp
            total_fp += fp
            total_fn += fn

            if pred_items != gold_items:
                sample_exact = False

        if sample_exact:
            exact_match_count += 1

        if show_progress and (i + 1) % 10 == 0:
            print(f"evaluated {i + 1}/{n}")

    precision = safe_div(total_tp, total_tp + total_fp)
    recall = safe_div(total_tp, total_tp + total_fn)
    f1 = safe_div(2 * precision * recall, precision + recall)

    return {
        "samples": n,
        "parse_fail": parse_fail_count,
        "parse_success_rate": safe_div(n - parse_fail_count, n),
        "exact_match_accuracy": safe_div(exact_match_count, n),
        "micro_precision": precision,
        "micro_recall": recall,
        "micro_f1": f1,
    }


base_result = evaluate_base_exaone(
    test_dataset,
    show_progress=True,
)

print("\n==============================")
print("Base EXAONE Test Metrics")
print("==============================")
print("samples              :", base_result["samples"])
print("parse_fail           :", base_result["parse_fail"])
print("parse_success_rate   :", round(base_result["parse_success_rate"], 4))
print("exact_match_accuracy :", round(base_result["exact_match_accuracy"], 4))
print("micro_precision      :", round(base_result["micro_precision"], 4))
print("micro_recall         :", round(base_result["micro_recall"], 4))
print("micro_f1             :", round(base_result["micro_f1"], 4))

[transformers] The `check_model_inputs` decorator is deprecated in favor of `merge_with_config_defaults`.


[ERROR] `cache_position` is part of ExaoneModel.forward's signature, but not documented. Make sure to add it to the docstring of the function in /workspace/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py.
[ERROR] `cache_position` is part of ExaoneForCausalLM.forward's signature, but not documented. Make sure to add it to the docstring of the function in /workspace/.cache/huggingface/modules/transformers_modules/LGAI_hyphen_EXAONE/EXAONE_hyphen_3_dot_5_hyphen_2_dot_4B_hyphen_Instruct/ccce25bd39c141fe053e0bc75818a8f5fe962802/modeling_exaone.py.


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

/tmp/ipykernel_5677/4026542359.py:129: UserWarning: Torchaudio's I/O functions now support per-call backend dispatch. Importing backend implementation directly is no longer guaranteed to work. Please use `backend` keyword with load/save/info function, instead of calling the underlying implementation directly.
  if module is None or not hasattr(module, "create_causal_mask"):
[transformers] The remote code model you are currently using seems to expect `cache_position`. This arg has been removed from the Transformers library, and will stop being created in `generate` even for remote code models in a future release. Please open a PR on the remote code hub repo to remove any usage of `cache_position`.


[PATCH] EXAONE compatibility patch applied.
[LOAD] Base EXAONE loaded for evaluation only.
evaluated 10/100
evaluated 20/100
evaluated 30/100
evaluated 40/100
evaluated 50/100
evaluated 60/100
evaluated 70/100
evaluated 80/100
evaluated 90/100
evaluated 100/100

Base EXAONE Test Metrics
samples              : 100
parse_fail           : 0
parse_success_rate   : 1.0
exact_match_accuracy : 0.03
micro_precision      : 0.3515
micro_recall         : 0.4474
micro_f1             : 0.3937


In [10]:
import json
from IPython.display import display, Markdown


def inspect_base_prediction(idx=0):
    sample = test_dataset[idx]

    input_text = sample["messages"][1]["content"]
    gold_text = sample["messages"][-1]["content"]

    pred_raw = predict_base_exaone(input_text)

    try:
        pred_json = parse_prediction_json(pred_raw)
        parse_ok = True
        parse_error = None
    except Exception as e:
        pred_json = empty_masking_result()
        parse_ok = False
        parse_error = str(e)

    gold_json = normalize_gold_json(gold_text)

    rows = []
    for key in LABEL_CATS:
        pred_values = pred_json.get(key, [])
        gold_values = gold_json.get(key, [])

        pred_set = item_set(pred_values)
        gold_set = item_set(gold_values)

        status = "OK" if pred_set == gold_set else "DIFF"
        extra = sorted(pred_set - gold_set)
        missing = sorted(gold_set - pred_set)

        rows.append(
            "| {key} | {status} | `{pred}` | `{gold}` | `{extra}` | `{missing}` |".format(
                key=key,
                status=status,
                pred=json.dumps(pred_values, ensure_ascii=False),
                gold=json.dumps(gold_values, ensure_ascii=False),
                extra=json.dumps(extra, ensure_ascii=False),
                missing=json.dumps(missing, ensure_ascii=False),
            )
        )

    table = "\n".join([
        "| field | status | prediction | gold | extra_fp | missing_fn |",
        "|---|---:|---|---|---|---|",
        *rows,
    ])

    display(Markdown(f"## Test index `{idx}`"))

    display(Markdown("### 입력 원문"))
    display(Markdown(f"```json\n{input_text}\n```"))

    display(Markdown("### 정답 원문"))
    display(Markdown(f"```json\n{gold_text}\n```"))

    display(Markdown("### Base EXAONE 추론 원문"))
    display(Markdown(f"```text\n{pred_raw}\n```"))

    display(Markdown("### 파싱 결과"))
    if parse_ok:
        display(Markdown(f"```json\n{json.dumps(pred_json, ensure_ascii=False, indent=2)}\n```"))
    else:
        display(Markdown(f"```text\nPARSE FAIL\n{parse_error}\n```"))

    display(Markdown("### 필드별 비교"))
    display(Markdown(table))


inspect_base_prediction(0)

## Test index `0`

### 입력 원문

```json
{"id": 367, "account_id": 228, "job_name": "안드로이드 개발자 채용", "education_level": "대졸 이상", "major": "무관", "career_level": "경력 3년 이상", "required_skill": ["mysql", "CSS", "python", "go"], "preferred_skill": ["redis", "mysql"], "main_task": "안드로이드 개발자 업무 전반", "hiring_reason": "", "work_type": "정규직", "status": "on_going", "created_at": "2026-04-01T09:00:00", "updated_at": "2026-04-15T09:00:00"}
```

### 정답 원문

```json
{"comp_name": [], "person_name": [], "address": [], "personal_info": [], "school_edu": [], "project_name": [], "jd_discrimination": []}
```

...

```text
```json
{
  "comp_name": ["안드로이드 개발자 채용"],
  "person_name": [],
  "address": [],
  "personal_info": [],
  "school_edu": ["대졸 이상", "무관"],
  "project_name": ["안드로이드 개발자 업무 전반"],
  "jd_discrimination": []
}
```

### 설명:
- **comp_name**: 채용 공고에서 명시된 회사명 또는 직무명인 "안드로이드 개발자 채용"이 포함되었습니다.
- **person_name**: 지원자 본인의 이름이나 제3자의 실명 정보가 없으므로 빈 리스트입니다.
- **address**: 주소 정보가 없으므로 빈 리스트입니다.
- **personal_info**: 지원자의 고유식별정보나 연락처 등 민감한 개인 정보가 명시되지 않았습니다. 따라서 "대졸 이상", "무관" 등은 학력 정보로 간주되어 포함되었습니다.
- **school_edu**: 학력 수준인 "대졸 이상"과 "무관"이 포함되었습니다.
- **project_name**: 채용 공고에서 언급된 주요 업무 범위인 "안드로이드 개발자 업무 전반"이 포함되었습니다.
- **jd_discrimination**: 채용 공고 본문에 차별 소지가 있는 내용이 명시되지 않았으므로 빈 리스트입니다.
```

### 파싱 결과

```json
{
  "comp_name": [
    "안드로이드 개발자 채용"
  ],
  "person_name": [],
  "address": [],
  "personal_info": [],
  "school_edu": [
    "대졸 이상",
    "무관"
  ],
  "project_name": [
    "안드로이드 개발자 업무 전반"
  ],
  "jd_discrimination": []
}
```

### 필드별 비교

| field | status | prediction | gold | extra_fp | missing_fn |
|---|---:|---|---|---|---|
| comp_name | DIFF | `["안드로이드 개발자 채용"]` | `[]` | `["안드로이드 개발자 채용"]` | `[]` |
| person_name | OK | `[]` | `[]` | `[]` | `[]` |
| address | OK | `[]` | `[]` | `[]` | `[]` |
| personal_info | OK | `[]` | `[]` | `[]` | `[]` |
| school_edu | DIFF | `["대졸 이상", "무관"]` | `[]` | `["대졸 이상", "무관"]` | `[]` |
| project_name | DIFF | `["안드로이드 개발자 업무 전반"]` | `[]` | `["안드로이드 개발자 업무 전반"]` | `[]` |
| jd_discrimination | OK | `[]` | `[]` | `[]` | `[]` |

In [11]:
from IPython.display import display, Markdown
import json


def show_base_wrong_examples(dataset_to_eval, max_examples=3, max_scan=None):
    shown = 0
    n = len(dataset_to_eval) if max_scan is None else min(len(dataset_to_eval), max_scan)

    for i in range(n):
        sample = dataset_to_eval[i]

        input_text = sample["messages"][1]["content"]
        gold_text = sample["messages"][-1]["content"]
        gold_json = normalize_gold_json(gold_text)

        pred_raw = predict_base_exaone(input_text)

        try:
            pred_json = parse_prediction_json(pred_raw)
            parse_ok = True
            parse_error = ""
        except Exception as e:
            pred_json = empty_masking_result()
            parse_ok = False
            parse_error = str(e)

        is_wrong = not parse_ok

        rows = []
        for key in LABEL_CATS:
            pred_values = pred_json.get(key, [])
            gold_values = gold_json.get(key, [])

            pred_set = item_set(pred_values)
            gold_set = item_set(gold_values)

            extra_fp = sorted(pred_set - gold_set)
            missing_fn = sorted(gold_set - pred_set)

            if extra_fp or missing_fn:
                is_wrong = True

            status = "OK" if not extra_fp and not missing_fn else "DIFF"

            rows.append(
                "| {key} | {status} | `{pred}` | `{gold}` | `{extra}` | `{missing}` |".format(
                    key=key,
                    status=status,
                    pred=json.dumps(pred_values, ensure_ascii=False),
                    gold=json.dumps(gold_values, ensure_ascii=False),
                    extra=json.dumps(extra_fp, ensure_ascii=False),
                    missing=json.dumps(missing_fn, ensure_ascii=False),
                )
            )

        if not is_wrong:
            continue

        table = "\n".join([
            "| field | status | prediction | gold | extra_fp | missing_fn |",
            "|---|---:|---|---|---|---|",
            *rows,
        ])

        display(Markdown(f"## Wrong Example `{shown + 1}` / Test index `{i}`"))

        display(Markdown("### 입력"))
        display(Markdown(f"```json\n{input_text}\n```"))

        display(Markdown("### Base EXAONE 추론 원문"))
        display(Markdown(f"```text\n{pred_raw}\n```"))

        display(Markdown("### 정답 원문"))
        display(Markdown(f"```json\n{gold_text}\n```"))

        display(Markdown("### 파싱 상태"))
        if parse_ok:
            display(Markdown("```text\nPARSE OK\n```"))
        else:
            display(Markdown(f"```text\nPARSE FAIL\n{parse_error}\n```"))

        display(Markdown("### 필드별 비교"))
        display(Markdown(table))

        shown += 1
        if shown >= max_examples:
            break

    if shown == 0:
        display(Markdown("## 틀린 예시 없음"))


show_base_wrong_examples(test_dataset, max_examples=3)

## Wrong Example `1` / Test index `0`

### 입력

```json
{"id": 367, "account_id": 228, "job_name": "안드로이드 개발자 채용", "education_level": "대졸 이상", "major": "무관", "career_level": "경력 3년 이상", "required_skill": ["mysql", "CSS", "python", "go"], "preferred_skill": ["redis", "mysql"], "main_task": "안드로이드 개발자 업무 전반", "hiring_reason": "", "work_type": "정규직", "status": "on_going", "created_at": "2026-04-01T09:00:00", "updated_at": "2026-04-15T09:00:00"}
```

### Base EXAONE 추론 원문

```text
```json
{
  "comp_name": ["안드로이드 개발자 채용"],
  "person_name": [],
  "address": [],
  "personal_info": [],
  "school_edu": ["대졸 이상", "무관"],
  "project_name": ["안드로이드 개발자 업무 전반"],
  "jd_discrimination": []
}
```

### 설명:
- **comp_name**: 채용 공고에서 명시된 회사명 또는 직무명인 "안드로이드 개발자 채용"이 포함되었습니다.
- **person_name**: 지원자 본인의 이름이나 제3자의 실명 정보가 없으므로 빈 리스트입니다.
- **address**: 주소 정보가 없으므로 빈 리스트입니다.
- **personal_info**: 지원자의 고유식별정보나 연락처 등 민감한 개인 정보가 명시되지 않았습니다. 따라서 "대졸 이상", "무관" 등은 학력 정보로 간주되어 포함되었습니다.
- **school_edu**: 학력 수준인 "대졸 이상"과 "무관"이 포함되었습니다.
- **project_name**: 채용 공고에서 언급된 주요 업무 범위인 "안드로이드 개발자 업무 전반"이 포함되었습니다.
- **jd_discrimination**: 채용 공고 본문에 차별 소지가 있는 내용이 명시되지 않았으므로 빈 리스트입니다.
```

...

```json
{"comp_name": ["쿠팡", "현대백화점"], "person_name": ["배지우"], "address": [], "personal_info": [], "school_edu": ["부산대학교 전자공학과", "엘리스"], "project_name": ["올인원 ERP 고도화"], "jd_discrimination": []}
```

### 파싱 상태

```text
PARSE OK
```

### 필드별 비교

| field | status | prediction | gold | extra_fp | missing_fn |
|---|---:|---|---|---|---|
| comp_name | DIFF | `["부산대학교 전자공학과"]` | `["쿠팡", "현대백화점"]` | `["부산대학교 전자공학과"]` | `["쿠팡", "현대백화점"]` |
| person_name | OK | `["배지우"]` | `["배지우"]` | `[]` | `[]` |
| address | OK | `[]` | `[]` | `[]` | `[]` |
| personal_info | DIFF | `["bachelor", "부산대학교 전자공학과"]` | `[]` | `["bachelor", "부산대학교 전자공학과"]` | `[]` |
| school_edu | DIFF | `["부산대학교 전자공학과"]` | `["부산대학교 전자공학과", "엘리스"]` | `[]` | `["엘리스"]` |
| project_name | DIFF | `[]` | `["올인원 ERP 고도화"]` | `[]` | `["올인원 ERP 고도화"]` |
| jd_discrimination | OK | `[]` | `[]` | `[]` | `[]` |

In [12]:
# base 모델로 test 3번 데이터 추론 결과
idx = 3
sample = test_dataset[idx]

input_text = sample["messages"][1]["content"]
gold_text = sample["messages"][-1]["content"]
input_obj = json.loads(input_text)

base_pred = predict_base_exaone(input_text)

print("test_idx:", idx)
print("original_id:", input_obj.get("id"))
print("job_name:", input_obj.get("job_name"))

print("\n=== 입력 ===")
print(input_text)

print("\n=== Base EXAONE 추론 원문 ===")
print(base_pred)

print("\n=== 정답 원문 ===")
print(gold_text)

test_idx: 3
original_id: 380
job_name: QA 엔지니어 채용

=== 입력 ===
{"id": 380, "account_id": 239, "job_name": "QA 엔지니어 채용", "education_level": "초대졸 이상", "major": "컴퓨터공학 또는 관련 학과", "career_level": "경력무관", "required_skill": ["node.js", "JS", "typescript"], "preferred_skill": [], "main_task": "QA 엔지니어 업무 전반", "hiring_reason": "", "work_type": "계약직", "status": "closed", "created_at": "2026-04-01T09:00:00", "updated_at": "2026-04-15T09:00:00"}

=== Base EXAONE 추론 원문 ===
```json
{
  "comp_name": ["QA 엔지니어 채용"],
  "person_name": [],
  "address": [],
  "personal_info": [],
  "school_edu": ["컴퓨터공학 또는 관련 학과"],
  "project_name": [],
  "jd_discrimination": []
}
```

=== 정답 원문 ===
{"comp_name": [], "person_name": [], "address": [], "personal_info": [], "school_edu": [], "project_name": [], "jd_discrimination": []}
